# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/madaraf/Starter-Notebooks-Assignment-Flyrank-/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

# Ensure repository root is in Python import path
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "Starter CSV not found."
print("Setup complete and ml_utils accessible.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Setup complete and ml_utils accessible.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

# **Task type: Ranking / Scoring.**

I'm not just trying to sort pages into "declining" vs "not declining" buckets. I want to give every page a score, then sort all pages from highest score to lowest score — so an editor gets a prioritized list, not just two piles.

This matches how the problem actually gets used: nobody reviews all 30,000 pages. An editor looks at the top of a list. A plain classification model (yes/no) doesn't naturally give you an order — a ranking/scoring model does.

This also matches the reference pipeline already in the repo: it builds a baseline_refresh_score and a final_refresh_score, then sorts pages by that score. I'm following the same shape.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

# **Target: is_declining_label (1 if trend_direction == "down", else 0).**

This isn't a column that comes in the raw CSV — it's a proxy I derive with one line of code from the existing trend_direction column:

```
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
```



I'm calling it a proxy, not a perfect answer, because "traffic dropped 20%+ over the last 30 days vs the previous 30 days" is not the same thing as "this page genuinely needs editorial attention" — it's just the closest observable stand-in I have. A page could dip for a season and recover on its own, or decline for reasons a refresh won't fix. I'll keep that limitation in mind when I report results later.

Because the label is computed directly from trend_direction (and trend_direction is computed from trend_pct), neither of those two columns can ever be used as a feature — that would be leakage, not learning.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

# **Metric: Precision@50.**

Of the top 50 pages my ranking puts at the top, what fraction actually turned out to be declining (is_declining_label == 1)?

I'm choosing 50 (instead of 20 or 100) for two reasons:



1.   It roughly matches a realistic weekly review capacity for an editor.
2.   It's the exact number the already-committed baseline and model results use (baseline_precision_at_50, Precision@50 = 0.240 for the rule, 0.740 for the random forest) — so anything I build later is directly comparable to real, already-verified numbers instead of a metric I invented myself.

I'll also print the base rate (54.2% of all pages are labeled declining) next to any Precision@50 number I report, so a reader can tell how much of my score is real skill versus just the label being fairly common.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

# **Unit of analysis: one row = one content page.**

Not a client, not a day, not a query — a single published page, identified by **content_id**, described by its trailing-90-day search and engagement metrics. Below, I load the starter slice and print a few rows to prove this concretely, rather than just asserting it.

In [21]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Rows: {len(df):,}   Columns: {df.shape[1]}")
print(f"Unique content_id values: {df['content_id'].nunique():,}  "
      f"(should equal row count if one row = one page)")

# Show what "one row" actually looks like
df[["content_id", "client_id", "impressions_90d", "avg_position",
    "ctr", "days_since_last_update", "word_count", "trend_direction"]].head(5)

Rows: 30,000   Columns: 44
Unique content_id values: 30,000  (should equal row count if one row = one page)


,content_id,client_id,impressions_90d,avg_position,ctr,days_since_last_update,word_count,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,10.6,0.76,20,3221.0,down
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,0.05,25,2481.0,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,0.09,20,3515.0,down
3,content_331d6c4de07b,client_19581e27de,11751,6.2,0.49,22,NaN,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,0.13,14,2803.0,down


In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple hand-written rule (e.g. "flag a page if it's old AND still gets traffic") is easy to write, but it's blunt — it lumps together several very different reasons a page could need attention (declining traffic, weak CTR at a good position, thin content, low engagement) into one bucket, and it can't weigh which signal matters more for which page.

The evidence below, pulled from the reference pipeline already run in this repo, shows the gap concretely

*   The hand-written baseline rule gets Precision@50 = 0.240 — only ~12 of its top 50 picks are actually declining.

*   A random forest trained on the same observable signals gets Precision@50 = 0.740 — ~37 of 50 are correct, about 3x better, on the exact same pages and the exact same metric.
*   A tempting single-column shortcut — just rank by **search_volume** — wouldn't have worked either: it correlates with real traffic (**impressions_90d**) at only 0.001, essentially zero.

This tells me the real pattern is spread across several weak, tangled signals (position, freshness, CTR, engagement, age) that interact in ways a single rule can't capture, but a model — trained on the same data — clearly can.



In [23]:
!PYTHONPATH=scripts python scripts/run_all.py


▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship-starter/flyrank-ml-

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import json

# Load the already-computed baseline vs. model results from the reference pipeline
try:
    res = json.load(open("outputs/model_results.json"))
    base = res["baseline"]["baseline_precision_at_50"]
    rf = res["models"]["random_forest"]["precision_at_50"]
    print(f"Baseline rule  Precision@50: {base:.3f}  (~{round(base*50)} of top 50 correct)")
    print(f"Random forest  Precision@50: {rf:.3f}  (~{round(rf*50)} of top 50 correct)")
    print(f"Lift: {rf/base:.1f}x")
except FileNotFoundError:
    print("outputs/model_results.json not found — run `python scripts/run_all.py` once first.")

# Check whether a naive single-signal shortcut would have worked
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"\nCorrelation(search_volume, impressions_90d): {corr:.3f}  "
      "(near zero -> a one-column rule wouldn't have worked either)")

Baseline rule  Precision@50: 0.240  (~12 of top 50 correct)
Random forest  Precision@50: 0.740  (~37 of top 50 correct)
Lift: 3.1x

Correlation(search_volume, impressions_90d): 0.001  (near zero -> a one-column rule wouldn't have worked either)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.